# Customer Churn — Prediction & Drivers

:::{admonition} How to use this notebook
:class: tip dropdown
This is a working template for a project page. The structure — question, data, EDA,
model, evaluation, recommendation — is what a hiring manager scans for. Swap in your
own dataset and keep the headings.

The data here is synthetic (`data/make_sample_data.py`) so the page rebuilds anywhere.
:::

## The question

A subscription business loses ~17% of its base each year. Two things are needed:

1. **A ranked list** of who is most likely to leave next, so retention can target it.
2. **The drivers**, so the product team can fix the cause rather than paper over it.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

pd.set_option("display.precision", 3)
plt.rcParams.update({
    "figure.figsize": (7, 4),
    "figure.dpi": 120,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "grid.alpha": 0.25,
    "font.size": 10,
})

INK, ACCENT = "#2b3a55", "#c8553d"

# The book build executes notebooks from the project root; JupyterLab runs
# them from this folder. Resolve the data directory for both cases.
DATA = Path("data") if Path("data").is_dir() else Path("../data")

## The data

One row per subscriber, with plan, contract, usage, and support load. `churned` is the
target: 1 if they cancelled in the observation window.

In [ ]:
df = pd.read_csv(DATA / "subscribers.csv")
print(f"{len(df):,} subscribers · {df.churned.mean():.1%} churned")
df.head()

In [ ]:
df.describe(include="all").T

## Exploratory analysis

Two questions first: **who** churns, and **how much** each factor moves the rate.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 3.6))

by_contract = df.groupby("contract").churned.mean().sort_values()
axes[0].barh(by_contract.index, by_contract.values, color=[INK, ACCENT])
axes[0].set_title("Churn rate by contract type")
axes[0].xaxis.set_major_formatter(lambda x, _: f"{x:.0%}")

bins = pd.cut(df.tenure_months, [0, 6, 12, 24, 36, 60])
by_tenure = df.groupby(bins, observed=True).churned.mean()
axes[1].plot(range(len(by_tenure)), by_tenure.values, "o-", color=ACCENT)
axes[1].set_xticks(range(len(by_tenure)), [str(i) for i in by_tenure.index], rotation=20)
axes[1].set_title("Churn rate by tenure (months)")
axes[1].yaxis.set_major_formatter(lambda y, _: f"{y:.0%}")

fig.tight_layout()

Monthly contracts churn far harder than annual ones, and risk concentrates in
the first year. Both point at onboarding rather than pricing.

In [ ]:
summary = (
    df.groupby(["plan", "contract"])
      .agg(subscribers=("customer_id", "size"),
           churn_rate=("churned", "mean"),
           avg_tickets=("support_tickets", "mean"))
      .sort_values("churn_rate", ascending=False)
)
summary.style.format({"churn_rate": "{:.1%}", "avg_tickets": "{:.2f}"})

## A baseline model

Logistic regression, written out with NumPy so this notebook needs no extra
dependencies. **Swap in `sklearn.linear_model.LogisticRegression` for real work** —
the point here is the workflow, not the solver.

In [ ]:
FEATURES = ["tenure_months", "monthly_charge", "support_tickets", "logins_last_30d"]

X = df[FEATURES].to_numpy(float)
X = np.column_stack([X, (df.contract == "Annual").to_numpy(float)])
names = FEATURES + ["contract_annual"]
y = df.churned.to_numpy(float)

# Standardise, then hold out 30% for evaluation.
mu, sd = X.mean(0), X.std(0)
Xs = np.column_stack([np.ones(len(X)), (X - mu) / sd])

rng = np.random.default_rng(0)
idx = rng.permutation(len(X))
cut = int(0.7 * len(X))
train, test = idx[:cut], idx[cut:]


def fit(X, y, lr=0.5, epochs=3000):
    w = np.zeros(X.shape[1])
    for _ in range(epochs):
        p = 1 / (1 + np.exp(-X @ w))
        w -= lr * X.T @ (p - y) / len(y)
    return w


w = fit(Xs[train], y[train])
p_test = 1 / (1 + np.exp(-Xs[test] @ w))
print(f"trained on {len(train):,} rows, evaluating on {len(test):,}")

In [ ]:
coefs = (
    pd.DataFrame({"feature": names, "coefficient": w[1:]})
      .assign(odds_ratio=lambda d: np.exp(d.coefficient))
      .sort_values("coefficient", key=abs, ascending=False)
      .reset_index(drop=True)
)
coefs

## Evaluation

Accuracy is the wrong metric on a 17%-positive problem — predicting "nobody churns"
scores 83%. Ranking quality (AUC) and lift in the top decile are what a retention
team can act on.

In [ ]:
def auc(y_true, scores):
    order = np.argsort(scores)
    ranks = np.empty(len(scores)); ranks[order] = np.arange(1, len(scores) + 1)
    n1, n0 = y_true.sum(), (1 - y_true).sum()
    return (ranks[y_true == 1].sum() - n1 * (n1 + 1) / 2) / (n1 * n0)


y_test = y[test]
base = y_test.mean()
top = p_test >= np.quantile(p_test, 0.9)

print(f"AUC                    {auc(y_test, p_test):.3f}")
print(f"Base churn rate        {base:.1%}")
print(f"Top-decile churn rate  {y_test[top].mean():.1%}")
print(f"Lift                   {y_test[top].mean() / base:.1f}x")

In [ ]:
deciles = pd.qcut(p_test, 10, labels=False)
rates = pd.Series(y_test).groupby(deciles).mean()

fig, ax = plt.subplots()
ax.bar(rates.index + 1, rates.values, color=INK)
ax.bar(10, rates.iloc[-1], color=ACCENT)
ax.axhline(base, color="grey", ls="--", lw=1)
ax.annotate("overall rate", (0.7, base), xytext=(0.7, base + 0.03), fontsize=9, color="grey")
ax.set(xlabel="Risk decile (10 = highest predicted risk)",
       ylabel="Actual churn rate", title="Model ranking on held-out subscribers")
ax.set_xticks(range(1, 11))
ax.yaxis.set_major_formatter(lambda v, _: f"{v:.0%}")
fig.tight_layout()

## What this means

:::{important} Recommendation
Target the top risk decile — it contains a disproportionate share of churners, so a
fixed retention budget goes roughly 2–3x further than untargeted outreach.
:::

- **Support load is the strongest movable driver.** Each additional ticket raises churn
  odds materially; ticket volume is a leading indicator worth alerting on.
- **Annual contracts cut risk sharply.** An upgrade offer at month 3 is cheaper than
  win-back at month 12.
- **Risk concentrates in the first year**, which makes this an onboarding problem more
  than a pricing one.

### Limitations

- Synthetic data — replace with production extracts before drawing real conclusions.
- No time-based split; a production model must be backtested forward in time.
- Cost-weighted thresholds (contact cost vs. retained lifetime value) still need setting.

### Next steps

1. Re-fit on real data with a forward-in-time backtest.
2. Add behavioural features: feature adoption, billing failures, NPS responses.
3. Ship the scores to the CRM weekly and A/B test the intervention.